# NeuralAI D17 — DPO Training on Colab GPU

Trains `train_d17_dpo.py` (SmolLM2-360M-Instruct + v16 adapter → v17 DPO adapter).

**CRITICAL:** The script uses RELATIVE paths (`./checkpoints/...`, `data/...`, `train_d17_dpo.py`).
Every cell below `%cd`s into the cloned repo before touching files. Never run the script from `/content`.

Steps:
1. Run **Setup** (mounts GPU, clones/pulls repo, installs deps).
2. Run **Verify assets** (`%cd` into repo first).
3. Run **Train (GPU)** — force-patches the script to use CUDA (original falls back to CPU in Zo).
4. Run **Download v17** to pull `checkpoints/v17-dpo` back.

In [ ]:
# ===== SETUP =====
import os, subprocess, sys

# 1) GPU sanity (Runtime > Change runtime type > GPU — pick T4/A100/L4)
gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True)
print('GPU:\n', gpu.stdout.strip() or 'NO GPU — switch runtime to GPU')

# 2) Clone / refresh the NeuralAI repo (public)
REPO_DIR = '/content/NeuralAI'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','https://github.com/Subject-Emu-5259/NeuralAI.git', REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
print('repo at', REPO_DIR)

# 3) Install deps (transformers/peft/trl + bitsandbytes for 4-bit)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade',
                'transformers','peft','trl','datasets','bitsandbytes','accelerate'], check=True)
print('deps installed')

In [ ]:
# ===== VERIFY ASSETS (must be INSIDE repo) =====
%cd /content/NeuralAI
import os
for p in [
    'train_d17_dpo.py',
    'checkpoints/v2_model/adapter_model.safetensors',
    'checkpoints/v2_model/adapter_config.json',
    'data/train_dpo_v16_combined.jsonl',
]:
    print(('OK  ' if os.path.exists(p) else 'MISSING '), p)
assert os.path.exists('train_d17_dpo.py'), 'train_d17_dpo.py not found — repo not pulled? Run Setup cell again.'
assert os.path.exists('checkpoints/v2_model/adapter_model.safetensors'), 'v16 adapter missing'
assert os.path.exists('data/train_dpo_v16_combined.jsonl'), 'v16 dataset missing'
print('all assets present')

In [ ]:
# ===== TRAIN ON GPU =====
# The original script force-sets cfg.device = cpu for Zo (no CUDA there).
# On Colab we WANT the GPU, so we remove that line before running.
%cd /content/NeuralAI
src = open('train_d17_dpo.py').read()
# Defensive patch: drop the exact CPU-forcing line if present.
cpu_line = 'cfg.device = torch.device("cpu")\n'
if cpu_line in src:
    src = src.replace(cpu_line, '')
    print('patched: removed CPU-forcing line -> CUDA will be used if available')
else:
    print('NOTE: CPU-forcing line not found verbatim; device will follow torch.cuda.is_available()')
open('train_d17_dpo.py','w').write(src)

# Confirm torch sees the GPU before launching
import torch
print('cuda available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())

# Run the training
!python train_d17_dpo.py

In [ ]:
# ===== DOWNLOAD v17 ADAPTER BACK =====
%cd /content/NeuralAI
import os
out = 'checkpoints/v17-dpo'
print('exists:', os.path.isdir(out))
print('contents:', os.listdir(out) if os.path.isdir(out) else 'NONE')
# Colab: right-click checkpoints/v17-dpo in the file browser to download,
# or zip it for a single download:
!zip -r /content/v17-dpo.zip checkpoints/v17-dpo && echo 'zipped -> /content/v17-dpo.zip'